# 00 · System & ODD：一套 L4 系统究竟在解决什么问题？

你已经会深度学习；本章补的是自动驾驶的系统语言。先不要从模型名字开始，而要从一个可审计的任务开始：在明确的 **Operational Design Domain（ODD）** 内，车辆如何把多传感器观测变成安全动作？

整门课贯穿同一个 `urban cut-in` 场景：左侧车辆逐渐切入 ego lane，前方还有 lead vehicle。后续章节会复用本章的场景 ID、坐标约定、时间戳和 artifact。

本章合并了旧版 `00A` 的系统全景、旧版 `00` 的 ODD/系统契约，以及旧版 `00F` 的数据和发布证据入口。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, asdict

scene = build_urban_cut_in_scene(seed=7, timestamp_s=0.0, scene_id="urban_cut_in_demo")
print(scene.as_metadata())
print("camera points:", scene.camera_points.shape, "lidar points:", scene.lidar_points.shape)

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(scene.lidar_points[:, 0], scene.lidar_points[:, 1], s=5, alpha=0.25, label="LiDAR observation")
ax.scatter(*scene.cut_in_xy, color="crimson", s=80, label="cut-in actor")
ax.scatter(*scene.lead_xy, color="darkorange", s=80, label="lead actor")
ax.axhline(0, color="black", linewidth=0.7)
ax.set_aspect("equal")
ax.set(xlabel="ego x forward / m", ylabel="ego y left / m", title="The shared urban cut-in scene")
ax.legend()
plt.show()

## 1. ODD、ego、actor、scene 与模块接口

- **ego**：自车状态和坐标原点；
- **actor/agent**：其他交通参与者，至少需要位置、速度、尺寸和不确定性；
- **scene**：带时间戳的多传感器观测、地图/路线、ego pose 和 actor state；
- **ODD**：地理区域、道路类型、天气、光照、速度、地图和传感器健康度的约束；
- **模块接口**：输入字段、坐标系、时间 age、输出语义、latency 和退化动作。

`perception → tracking → prediction → planning → control` 是功能链，不等于必须使用五个独立神经网络。真正重要的是每个边界能否被测试、回放和降级。

In [ ]:
@dataclass(frozen=True)
class ODD:
    geography: str = "urban_mapped"
    max_speed_mps: float = 13.9
    night_allowed: bool = False
    max_rain_mm_h: float = 8.0
    min_sensor_health: float = 0.70
    max_localization_sigma_m: float = 0.80

odd = ODD()
scenario_table = pd.DataFrame([
    {"scenario_id": "urban_cut_in_demo", "geography": "urban_mapped", "speed_mps": 8.0,
     "night": False, "rain_mm_h": 0.0, "sensor_health": 0.94, "localization_sigma_m": 0.25},
    {"scenario_id": "night_glare", "geography": "urban_mapped", "speed_mps": 8.0,
     "night": True, "rain_mm_h": 0.0, "sensor_health": 0.74, "localization_sigma_m": 0.42},
    {"scenario_id": "stale_map", "geography": "urban_mapped", "speed_mps": 14.5,
     "night": False, "rain_mm_h": 4.0, "sensor_health": 0.88, "localization_sigma_m": 1.05},
])
checks = pd.DataFrame({
    "geography_ok": scenario_table.geography.eq(odd.geography),
    "speed_ok": scenario_table.speed_mps.le(odd.max_speed_mps),
    "light_ok": scenario_table.night.le(odd.night_allowed),
    "rain_ok": scenario_table.rain_mm_h.le(odd.max_rain_mm_h),
    "sensor_ok": scenario_table.sensor_health.ge(odd.min_sensor_health),
    "localization_ok": scenario_table.localization_sigma_m.le(odd.max_localization_sigma_m),
})
scenario_table["in_odd"] = checks.all(axis=1)
display(scenario_table)
display(checks.mean().sort_values().rename("constraint pass rate").to_frame())

## 2. 把系统契约落到输入、时间与状态

一个可用的 sensor bundle 不只是几个 tensor。它要声明 `frame_id`、每个传感器的 timestamp age、缺失处理、最大 latency 和输出可以触发的状态。模型 confidence 高并不能覆盖 stale input。

In [ ]:
SENSOR_CONTRACT = {
    "required": {"camera", "lidar", "timestamp_s", "frame_id"},
    "frame_id": "base_link",
    "max_age_s": {"camera": 0.15, "lidar": 0.10},
    "max_latency_ms": 100.0,
}

def validate_bundle(bundle):
    issues = sorted(SENSOR_CONTRACT["required"] - set(bundle))
    if bundle.get("frame_id") != SENSOR_CONTRACT["frame_id"]:
        issues.append("frame mismatch")
    for sensor, max_age in SENSOR_CONTRACT["max_age_s"].items():
        if bundle.get(f"{sensor}_age_s", np.inf) > max_age:
            issues.append(f"{sensor} is stale")
    if bundle.get("latency_ms", np.inf) > SENSOR_CONTRACT["max_latency_ms"]:
        issues.append("latency budget exceeded")
    return {"valid": not issues, "issues": issues}

valid = {"camera": [], "lidar": [], "timestamp_s": 0.0, "frame_id": "base_link",
         "camera_age_s": 0.04, "lidar_age_s": 0.03, "latency_ms": 62.0}
stale = {**valid, "lidar_age_s": 0.24, "latency_ms": 128.0}
print("valid:", validate_bundle(valid))
print("stale:", validate_bundle(stale))

artifact = {
    "scenario": scene.as_metadata(),
    "odd": asdict(odd),
    "sensor_contract": SENSOR_CONTRACT,
    "domain_checkpoint": {
        "next": "01_sensors_geometry.ipynb",
        "question": "how do raw camera/LiDAR observations enter a common frame?",
    },
}
save_json_artifact("00_system_contract.json", artifact)
print("saved:", ARTIFACT_DIR / "00_system_contract.json")

### 练习与完成标准

1. 给 ODD 增加 `route_available` 和 `map_version`，并写一个 `DEGRADED` / `MINIMAL_RISK` 判定；
2. 写出一个“模型 confidence 很高但输入 stale”的反例；
3. 用自己的话解释：为什么 ODD、sensor contract 和模型 loss 是三种不同层级的对象？

完成后你应该能画出一条 `sensor bundle → representation → agent state → prediction → planner → safety` 链路，并为每个箭头说出输入、输出和失败模式。